# Sesión 2 · Práctica guiada: vuestras funciones de desigualdad

Esta práctica la hacemos juntos, paso a paso: el profesor escribe y explica cada celda, y tú la reproduces en tu notebook.

Vais a escribir, una a una, las funciones para medir la desigualdad de renta con la Encuesta de Condiciones de Vida, y a comprobar que funcionan.

Variables del fichero de hogares:

| Variable | Significado |
| --- | --- |
| `DB040` | Región (`ES30` Madrid, `ES41` Castilla y León, `ES61` Andalucía…) |
| `DB090` | Peso del hogar: a cuántos hogares representa |
| `HY020` | Renta disponible del hogar en un año (euros) |
| `HX040` | Miembros del hogar |
| `HX240` | Unidades de consumo |

> El fichero `datos/ecv_hogares.csv` es **simulado** (ver `datos/ORIGEN.md`). Los microdatos reales se descargan gratis de la web del INE.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

In [ ]:
import numpy as np
import pandas as pd

ecv = pd.read_csv("datos/ecv_hogares.csv")
print(ecv.shape)
ecv.head(3)

## Ejercicio 1 · Preparar los arrays

1. Calculad la renta equivalente de cada hogar: `HY020` dividido por `HX240`.
2. Cada persona del hogar «recibe» esa renta, así que el peso para hablar de *personas* es `DB090` por `HX040`.
3. Quedaos solo con los hogares de renta positiva (en `renta`, en `peso` y en `region`).

In [ ]:
# TODO 1
...

print(len(renta), round(renta.mean()))
assert (renta > 0).all() and len(renta) == len(peso) == len(region)

## Ejercicio 2 · Media simple y media ponderada

Calculad la renta equivalente media sin ponderar y ponderada. ¿Cuánto cambia?

In [ ]:
# TODO 2
...

print(round(media_simple), round(media_ponderada))
assert media_simple != media_ponderada

## Ejercicio 3 · La función `gini`

Escribid la función con su *docstring*, usando la fórmula vista en teoría y **sin bucles**:

$$G = \frac{2\sum_i i\,x_i}{n\sum_i x_i} - \frac{n+1}{n}$$

In [ ]:
# TODO 3
...


assert abs(gini(np.full(50, 10.0))) < 1e-9          # todos iguales -> 0
assert abs(gini(np.array([0, 0, 0, 100])) - 0.75) < 1e-9
print("Gini sin ponderar:", round(gini(renta), 3))

## Ejercicio 4 · Gini con pesos

Con pesos, se ordenan rentas y pesos por renta y se calcula:

$$G = 1 - \sum_i (p_i - p_{i-1})\,(L_i + L_{i-1})$$

donde $p_i$ es la proporción acumulada de población y $L_i$ la proporción acumulada de renta. La función está empezada: completad las dos líneas que faltan.

In [ ]:
def gini_ponderado(rentas, pesos):
    """Índice de Gini teniendo en cuenta el peso de cada observación."""
    orden = np.argsort(rentas)                      # posiciones que ordenan las rentas
    x, w = rentas[orden], pesos[orden]

    # TODO 4a: proporción acumulada de población (suma acumulada de w dividida por el total de w)
    ...
    # TODO 4b: proporción acumulada de renta (suma acumulada de w * x dividida por su total)
    ...

    p = np.insert(p, 0, 0)                          # añadimos el punto de partida (0, 0)
    L = np.insert(L, 0, 0)
    return 1 - np.sum(np.diff(p) * (L[1:] + L[:-1]))


# con pesos iguales debe dar (casi) lo mismo que la función sin pesos
assert abs(gini_ponderado(renta, np.ones(len(renta))) - gini(renta)) < 0.001
print("Gini ponderado:", round(gini_ponderado(renta, peso), 3))

## Ejercicio 5 · Reutilizar: un resultado por región

Con un bucle sobre las regiones, construid un diccionario `{region: gini}` con el Gini ponderado de cada una. Después mostrad las tres regiones más desiguales.

Pista: `np.unique(region)` da la lista de regiones, y `region == "ES30"` es la máscara de los hogares de Madrid.

In [ ]:
# TODO 5
...

print(gini_por_region)
print(mas_desiguales)
assert len(gini_por_region) == len(np.unique(region))

## Ejercicio 6 · Una función que resume

Escribid `resumen_desigualdad(rentas, pesos)`, que devuelva un diccionario con la media ponderada, la mediana sin ponderar y el Gini ponderado, todo redondeado. Usadla para comparar Madrid (`ES30`) y Andalucía (`ES61`).

In [ ]:
# TODO 6
...


for codigo in ["ES30", "ES61"]:
    m = region == codigo
    print(codigo, resumen_desigualdad(renta[m], peso[m]))

assert set(resumen_desigualdad(renta, peso)) == {"media", "mediana", "gini"}

## Para cerrar

- ¿Qué decisiones habéis tomado por el camino (rentas negativas, pesos, unidades de consumo)? ¿Dónde han quedado escritas?
- Guardad estas funciones: las reutilizaréis en el **entregable 1** y, en la sesión 4, las moveremos a un fichero propio.